In [62]:
import os
import re
import sys
import numpy as np
import pandas as pd
import warnings
import logging
import matplotlib.pyplot as plt


LOG_CSV_EXPECTED = "/data/ros2/ros2_ws2/arm_bot/src/scripts/controller/dnn_controller/logs/expected_trajectory_1777379133.csv"
LOG_CSV_ACTUAL = "/data/ros2/ros2_ws2/arm_bot/src/scripts/controller/dnn_controller/logs/actual_movement_1777379133.csv"

def load_dataset_file(filepath):
    """
    Load the dataset CSV file (move_q2_traj format).
    Format: Each row is a variable (dp1, etc), columns are time steps.
    """
    data = pd.read_csv(filepath, index_col=0)
    
    # print(data.head())  # Print the first few rows to verify the structure
    
    # Time values are in the column names, convert to numpy array
    time = data.columns.astype(float).values
    
    # Extract relevant arrays by row index, convert to numpy arrays
    pos1 = data.loc['dp1'].values
    pos2 = data.loc['dp2'].values
    pos3 = data.loc['dp3'].values
    vel1 = data.loc['dv1'].values
    vel2 = data.loc['dv2'].values
    vel3 = data.loc['dv3'].values
    
    return time, pos1, pos2, pos3, vel1, vel2, vel3

ex_t, ex_dp1, ex_dp2, ex_dp3, ex_dv1, ex_dv2, ex_dv3 = load_dataset_file(LOG_CSV_EXPECTED)
ac_t, ac_dp1, ac_dp2, ac_dp3, ac_dv1, ac_dv2, ac_dv3 = load_dataset_file(LOG_CSV_ACTUAL)

# Plot positions
plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
plt.plot(ex_t, ex_dp1, label='ex Position 1')
plt.plot(ex_t, ex_dp2, label='ex Position 2')
plt.plot(ex_t, ex_dp3, label='ex Position 3')
plt.title('Joint Positions over Time')
plt.xlabel('Time (s)')
plt.ylabel('Position (rad)')
plt.legend()



# Plot velocities
plt.subplot(2, 1, 2)
plt.plot(ac_t, ac_dp1, label='ac Position 1')
plt.plot(ac_t, ac_dp2, label='ac Position 2')
plt.plot(ac_t, ac_dp3, label='ac Position 3')
plt.title('Joint Velocities over Time')
plt.xlabel('Time (s)')
plt.ylabel('Velocity (rad/s)')
plt.legend()

plt.tight_layout()
plt.show()
plt.savefig('joint_data_plots.png')

In [65]:
import pandas as pd
import re
import os

dataset_dir_path = "/data/ros2/ros2_ws2/arm_bot/src/scripts/Joint_states"

print(f'Processing dataset directory: {dataset_dir_path}')
count = 0
for filename in os.listdir(dataset_dir_path):
    # print(f'Checking file: {filename}')
    match = re.search(r'path_(\d+)_joint_states.csv$', filename)
    if match and int(match.group(1)) <= 450 and int(match.group(1)) >= 81:
        log_file_path = os.path.join(dataset_dir_path, filename)
        print(f'Processing log file: {filename}')

        df = pd.read_csv(log_file_path, index_col=0)

        # Extract relevant arrays by row index, convert to numpy arrays
        pos1 = df.loc['dp1'].values
        pos2 = df.loc['dp2'].values
        pos3 = df.loc['dp3'].values

        max_pos1_deg = max(pos1)* 180 / 3.141592653589793
        min_pos1_deg = min(pos1)* 180 / 3.141592653589793
        max_pos2_deg = max(pos2)* 180 / 3.141592653589793
        min_pos2_deg = min(pos2)* 180 / 3.141592653589793
        max_pos3_deg = max(pos3)* 180 / 3.141592653589793
        min_pos3_deg = min(pos3)* 180 / 3.141592653589793
        # convert the position values from radians to degrees
        if {float(max_pos1_deg) > 100 or float(min_pos1_deg) < -100}:
            print(f'joint 1 Max position: {max_pos1_deg}, Min position: {min_pos1_deg}')
            print(f'joint 2 Max position: {max_pos2_deg}, Min position: {min_pos2_deg}')
            print(f'joint 3 Max position: {max_pos3_deg}, Min position: {min_pos3_deg}')
            count += 1
    # else:
    #     print(f'Skipping file: {filename} - does not match pattern or path ID out of range')
print(f'Total files with joint 1 position exceeding ±80 degrees: {count}')

Processing dataset directory: /data/ros2/ros2_ws2/arm_bot/src/scripts/Joint_states
Processing log file: path_192_joint_states.csv
joint 1 Max position: 0.0, Min position: -37.66810651648346
joint 2 Max position: 44.999999999999986, Min position: 4.195389414120838
joint 3 Max position: 134.99999999999972, Min position: 108.99525256649055
Processing log file: path_281_joint_states.csv
joint 1 Max position: 0.0, Min position: -41.475623342487296
joint 2 Max position: 44.999999999999986, Min position: 7.880712266668345
joint 3 Max position: 134.99999999999972, Min position: 92.72473690342972
Processing log file: path_215_joint_states.csv
joint 1 Max position: 22.243959236684262, Min position: 0.0
joint 2 Max position: 44.999999999999986, Min position: 2.493272123898344
joint 3 Max position: 150.96052529083093, Min position: 134.99999999999972
Processing log file: path_298_joint_states.csv
joint 1 Max position: 0.0, Min position: -13.935767871363462
joint 2 Max position: 44.999999999999986,